# Phase 2: tokenizer + token shards (CPU session, ~2-4 h)
Run once. Accept the terms of `bigcode/the-stack-dedup` on huggingface.co first (it is gated).
Output: a private HF dataset `$HF_USERNAME/minillm-tokens` with uint16 shards + the tokenizer.

In [ ]:
# Kaggle: Add-ons -> Secrets: HF_TOKEN (write), HF_USERNAME, WANDB_API_KEY. Settings -> Internet ON.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"]:
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print(f"secret {name} not set")

GITHUB_REPO = "https://github.com/<your-github-username>/minillm.git"  # <- edit
%cd /kaggle/working
!git clone -q {GITHUB_REPO} minillm 2>/dev/null || (cd minillm && git pull -q)
%cd /kaggle/working/minillm
!pip install -q -e .

## Train the 32k byte-level BPE tokenizer

In [ ]:
!python -m MiniLLM.Data.Tokenizer --out artifacts/tokenizer --docs-per-source 150000

## Filter, dedup, tokenize, upload
If the session is tight on time, run English and code in two sessions with `--sources`.
`/kaggle/temp` has more disk than `/kaggle/working`.

In [ ]:
!python -m MiniLLM.Data.PrepareShards --tokenizer artifacts/tokenizer --out /kaggle/temp/tokens \
    --total-tokens 2.6e9 --push-to-hub $HF_USERNAME/minillm-tokens